# Clickbait Spoiler BERTScore Evaluation

In this notebook we evaluate the spoilers generated by our Ollama pipeline. We compare each generated spoiler with the gold spoiler from the dataset.


## What this notebook does

We load the generated results, clean the text fields, compute BERTScore for each generation method, and save the scored table.

BERTScore tells us how semantically similar a generated spoiler is to the gold spoiler. It does not prove that the spoiler is grounded in the article, so human checking is still useful later.


## Install libraries

In Colab we only need pandas and BERTScore.

In [1]:
%pip install -q bert-score pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.6 MB/s eta 0:00:00


## Import libraries and set paths

The input file comes from the generation notebook. The two output files are the full row-level BERTScore results and the small summary table.


In [2]:
from pathlib import Path

import pandas as pd
from bert_score import score as bert_score
from IPython.display import display

INPUT_PATH = Path("output/spoiler_generation_results.jsonl")
FULL_RESULTS_PATH = Path("output/spoiler_generation_results_bertscore.csv")
SUMMARY_PATH = Path("output/spoiler_bertscore_summary.csv")

FULL_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)


## Load generated spoiler results

We read the JSONL file that contains the gold spoilers and the three Ollama-generated spoiler variants.


In [3]:
results = pd.read_json(INPUT_PATH, lines=True)

print(f"Loaded rows: {len(results)}")
display(results.head())


Loaded rows: 300


,id,split,post_text,spoiler_type,article_context_short,gold_spoiler,generated_question,first_sentence_baseline,direct_llama_spoiler,question_llama_spoiler,type_aware_llama_spoiler
0,0af11f6b-c889-4520-9372-66ba25cb7657,train,"Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea",passage,It’ll be just like old times this weekend for Tom Brady and Wes Welker. Welker revealed Friday morning on a Miami ra...,how about that morning we go throw?,Did Wes Welker really get rejected by Tom Brady?,It’ll be just like old times this weekend for Tom Brady and Wes Welker.,Tom Brady asked Wes Welker to go throw footballs instead of having dinner.,Tom Brady had a different idea than dinner and instead invited Wes Welker to throw footballs with him.,Tom Brady had no intention of having dinner with Wes Welker and instead invited him to throw passes together.
1,b1a1f63d-8853-4a11-89e8-6b2952a393ec,train,NASA sets date for full recovery of ozone hole,phrase,2070 is shaping up to be a great year for Mother Earth. That's when NASA scientists are predicting the hole in the o...,2070,Is NASA finally fixing the ozone hole?,2070 is shaping up to be a great year for Mother Earth.,By 2070.,By 2070.,By 2070.
2,008b7b19-0445-4e16-8f9e-075b73f80ca4,train,This is what makes employees happy -- and it's not their paycheck,phrase,"Despite common belief, money isn't the key to employee happiness, new research finds. A study by hiring software pro...",intellectual stimulation,What makes employees happy?,"Despite common belief, money isn't the key to employee happiness, new research finds.","Intellectual stimulation accounts for 18.5% of job satisfaction, while money only accounts for 5.4%.",Intellectual stimulation.,Intellectual stimulation.
3,31ecf93c-3e21-4c80-949b-aa549a046b93,train,Passion is overrated — 7 work habits you need instead,multi,"It’s common wisdom. Near gospel really, and not just among young people and founders. Across generational lines, sen...","Purpose connects us to something bigger and in doing so makes us right sized be ruthless with your ""No’s."" Practice ...",Is passion really necessary for success?,It’s common wisdom.,Purpose deemphasizes the I and focuses on pursuing something outside yourself.,Not necessary.,Is passion really necessary for success?; No; Passion is overrated; Purpose deemphasizes the I; Purpose is about pur...
4,31b108a3-c828-421a-a4b9-cf651e9ac859,train,The perfect way to cook rice so that it's perfectly fluffy and NEVER sticks to the pan,phrase,"Boiling rice may seem simple, but there is a very fine line between under-cooked crunchy grains and mushy over-cooke...",in a rice cooker,Is there a secret to cooking rice that prevents it from sticking to the pan?,"Boiling rice may seem simple, but there is a very fine line between under-cooked crunchy grains and mushy over-cooke...",One piece of equipment is required: a rice cooker.,"Use a rice cooker or boil rice with a 1:1.5 ratio of rice to water, then simmer for 8 minutes and let rest for 15 mi...",One piece of equipment.


## Check required columns

Before scoring, we check that the notebook has the gold spoiler and all generated spoiler columns.


In [4]:
METHODS = {
    "direct": {
        "prediction": "direct_llama_spoiler",
        "precision": "bertscore_p_direct",
        "recall": "bertscore_r_direct",
        "f1": "bertscore_f1_direct",
    },
    "question": {
        "prediction": "question_llama_spoiler",
        "precision": "bertscore_p_question",
        "recall": "bertscore_r_question",
        "f1": "bertscore_f1_question",
    },
    "type_aware": {
        "prediction": "type_aware_llama_spoiler",
        "precision": "bertscore_p_type_aware",
        "recall": "bertscore_r_type_aware",
        "f1": "bertscore_f1_type_aware",
    },
}

BERTSCORE_COLUMNS = [
    "bertscore_p_direct",
    "bertscore_r_direct",
    "bertscore_f1_direct",
    "bertscore_p_question",
    "bertscore_r_question",
    "bertscore_f1_question",
    "bertscore_p_type_aware",
    "bertscore_r_type_aware",
    "bertscore_f1_type_aware",
]

required_columns = ["gold_spoiler"] + [settings["prediction"] for settings in METHODS.values()]
missing_columns = [column for column in required_columns if column not in results.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available.")


All required columns are available.


## Prepare text for scoring

We keep the original DataFrame unchanged as much as possible. For scoring, we create cleaned text versions where missing values become empty strings.

Empty gold spoilers or empty generated spoilers are not scored. Their BERTScore columns stay as `NaN`.


In [5]:
scored_results = results.copy()
reference_text = scored_results["gold_spoiler"].fillna("").astype(str).str.strip()

# Add empty BERTScore columns first. Valid rows will be filled below.
for column in BERTSCORE_COLUMNS:
    scored_results[column] = float("nan")


## Run BERTScore

We score each Ollama answer type separately. BERTScore returns precision, recall, and F1, and we store all three at row level.


In [6]:
summary_rows = []

for method_name, settings in METHODS.items():
    prediction_column = settings["prediction"]
    prediction_text = scored_results[prediction_column].fillna("").astype(str).str.strip()
    valid_rows = reference_text.ne("") & prediction_text.ne("")

    print(f"Scoring {method_name}: {int(valid_rows.sum())} rows")
    if not valid_rows.any():
        summary_rows.append(
            {
                "method": method_name,
                "evaluated_examples": 0,
                "average_bertscore_f1": float("nan"),
            }
        )
        continue

    precision, recall, f1 = bert_score(
        prediction_text[valid_rows].tolist(),
        reference_text[valid_rows].tolist(),
        lang="en",
        rescale_with_baseline=True,
        verbose=True,
    )

    scored_results.loc[valid_rows, settings["precision"]] = precision.cpu().numpy()
    scored_results.loc[valid_rows, settings["recall"]] = recall.cpu().numpy()
    scored_results.loc[valid_rows, settings["f1"]] = f1.cpu().numpy()

    summary_rows.append(
        {
            "method": method_name,
            "evaluated_examples": int(valid_rows.sum()),
            "average_bertscore_f1": float(f1.mean()),
        }
    )

print("BERTScore calculation finished.")


Scoring direct: 300 rows


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/9 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/5 [00:00<?, ?it/s]

done in 1.53 seconds, 196.06 sentences/sec
BERTScore calculation finished.


## Create the summary table

The summary table gives us one average BERTScore F1 value per method. This is useful for a quick comparison between direct, question-based, and type-aware generation.


In [7]:
bertscore_summary = pd.DataFrame(summary_rows)

display(bertscore_summary.round(4))


,method,evaluated_examples,average_bertscore_f1
0,direct,300,0.2860
1,question,300,0.3465
2,type_aware,300,0.3597


## Save outputs

We save the full result table with row-level BERTScore columns. We also save the compact summary table.


In [8]:
scored_results.to_csv(FULL_RESULTS_PATH, index=False)
bertscore_summary.to_csv(SUMMARY_PATH, index=False)

print(f"Full row-level results: {FULL_RESULTS_PATH}")
print(f"Summary table: {SUMMARY_PATH}")


Full row-level results: output/spoiler_generation_results_bertscore.csv
Summary table: output/spoiler_bertscore_summary.csv


## Show a few scored rows

In [9]:
preview_columns = [
    "post_text",
    "gold_spoiler",
    "direct_llama_spoiler",
    "question_llama_spoiler",
    "type_aware_llama_spoiler",
    "bertscore_f1_direct",
    "bertscore_f1_question",
    "bertscore_f1_type_aware",
]

display(scored_results[preview_columns].head())


,post_text,gold_spoiler,direct_llama_spoiler,question_llama_spoiler,type_aware_llama_spoiler,bertscore_f1_direct,bertscore_f1_question,bertscore_f1_type_aware
0,"Wes Welker Wanted Dinner With Tom Brady, But Patriots QB Had Better Idea",how about that morning we go throw?,Tom Brady asked Wes Welker to go throw footballs instead of having dinner.,Tom Brady had a different idea than dinner and instead invited Wes Welker to throw footballs with him.,Tom Brady had no intention of having dinner with Wes Welker and instead invited him to throw passes together.,0.094188,0.060840,0.055920
1,NASA sets date for full recovery of ozone hole,2070,By 2070.,By 2070.,By 2070.,0.563465,0.563464,0.563465
2,This is what makes employees happy -- and it's not their paycheck,intellectual stimulation,"Intellectual stimulation accounts for 18.5% of job satisfaction, while money only accounts for 5.4%.",Intellectual stimulation.,Intellectual stimulation.,0.279701,0.879601,0.879601
3,Passion is overrated — 7 work habits you need instead,"Purpose connects us to something bigger and in doing so makes us right sized be ruthless with your ""No’s."" Practice ...",Purpose deemphasizes the I and focuses on pursuing something outside yourself.,Not necessary.,Is passion really necessary for success?; No; Passion is overrated; Purpose deemphasizes the I; Purpose is about pur...,0.076177,0.122138,-0.020407
4,The perfect way to cook rice so that it's perfectly fluffy and NEVER sticks to the pan,in a rice cooker,One piece of equipment is required: a rice cooker.,"Use a rice cooker or boil rice with a 1:1.5 ratio of rice to water, then simmer for 8 minutes and let rest for 15 mi...",One piece of equipment.,0.531862,0.282619,0.216244
